# Phase- 6: Supervised Predictive Validation & Tiering (England_final_light)

Same design as the London version, adapted to the national model: **no severity feature**, and the
balanced radiance / resolution / employment weights. England-wide (~33,755 LSOAs).

1. **Validate**: build the score on the first 24 months, test whether it predicts the held-out
   final 12 months (ROC / AUC vs a past-crime baseline).
2. **Tier**: derive 4 tiers from three ROC-validated Youden cut-points (top 15% / 40% / 70% of
   future crime), then compare to the K-means k=4 tiers and map them.

**Reads:** `England/outputs/phase1`+`phase2`, `data/lsoa_nightlights_england.parquet`, the England
LSOA boundaries, `england_final_light/outputs/phase5`. **Writes:** `.../outputs/phase6/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report
print('Imports OK')

In [ ]:
# Loading Dataset
BASE = Path('Dataset Path')
P1   = BASE / 'england_final_light' / 'outputs' / 'phase1'
P2   = BASE / 'england_final_light' / 'outputs' / 'phase2'
P5   = BASE / 'england_final_light' / 'outputs' / 'phase5'
OUT  = BASE / 'england_final_light' / 'outputs' / 'phase6'
OUT.mkdir(parents=True, exist_ok=True)
SHP_FILE = BASE / 'data' / 'Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V5_-7203918579177758597' / 'LSOA_2021_EW_BGC_V5.shp'

SPLIT   = pd.Timestamp('2025-04-01')
TIER_LEVELS = [0.15, 0.40, 0.70]
LABEL_PCT   = 0.15

# england_final_light Phase 4 blended weights (no severity)
WEIGHTS = {'employment_deprivation': 0.2608, 'ntl_mean_radiance': 0.4519, 'resolution_rate': 0.2873}
LOG_FEATURES = ['ntl_mean_radiance']   # only the heavy-tailed feature is logged (as Phase 4)

RESOLVED_OUTCOMES = {
    'Suspect charged', 'Offender given a caution', 'Offender given a penalty notice',
    'Offender fined', 'Offender deported', 'Offender otherwise dealt with',
    'Suspect charged as part of another case', 'Local resolution',
    'Offender given a drugs possession warning', 'Offender given conditional discharge',
    'Offender given absolute discharge', 'Offender sent to prison',
    'Offender given suspended prison sentence', 'Offender given community sentence'}
TIER_COLOURS = {1: '#D62728', 2: '#FF7F0E', 3: '#2CA02C', 4: '#1F77B4'}
report = []
print('Config loaded (England, no severity).')

## 1. Load data; build static features

In [ ]:
crimes   = pd.read_parquet(P1 / 'phase1_crimes_england.parquet', columns=['Crime ID', 'Month', 'LSOA code'])
outcomes = pd.read_parquet(P1 / 'phase1_outcomes_england.parquet', columns=['Crime ID', 'Outcome type'])
fm       = pd.read_parquet(P2 / 'phase2_feature_matrix.parquet')
ntl      = pd.read_parquet(BASE / 'data' / 'lsoa_nightlights_england.parquet')[['lsoa21cd', 'ntl_mean_radiance']]

LSOAS  = fm[['lsoa21cd']].copy()
crimes = crimes[crimes['LSOA code'].isin(set(LSOAS['lsoa21cd']))].rename(columns={'LSOA code': 'lsoa21cd'})

static = fm[['lsoa21cd', 'employment_rank']].merge(ntl, on='lsoa21cd', how='left')
mr = static['employment_rank'].max()
static['employment_deprivation'] = mr + 1 - static['employment_rank']
static['ntl_mean_radiance'] = static['ntl_mean_radiance'].fillna(static['ntl_mean_radiance'].median())
print(f'Crimes (England): {len(crimes):,} | LSOAs: {len(LSOAS):,} | range {crimes["Month"].min():%b %Y}..{crimes["Month"].max():%b %Y}')

## 2. Temporal split

In [ ]:
train  = crimes[crimes['Month'] <  SPLIT].copy()
future = crimes[crimes['Month'] >= SPLIT].copy()
print(f'Train : {train["Month"].min():%b %Y}..{train["Month"].max():%b %Y} ({len(train):,})')
print(f'Future: {future["Month"].min():%b %Y}..{future["Month"].max():%b %Y} ({len(future):,})')
report += [f'Train 24mo: {len(train):,} crimes', f'Future 12mo: {len(future):,} crimes']

## 3. Rebuild the risk score on the 24-month window only
No severity feature here. Recompute the time-varying part (resolution rate) on the 24-month
window; employment deprivation and radiance are static; crime_count_24 is kept for the baseline.

In [ ]:
agg = train.groupby('lsoa21cd').agg(crime_count_24=('Crime ID', 'count')).reset_index()
tw = train[train['Crime ID'].notna()][['Crime ID', 'lsoa21cd']].merge(
        outcomes[outcomes['Crime ID'].notna()], on='Crime ID', how='left')
tw['resolved'] = tw['Outcome type'].apply(
        lambda x: any(r.lower() in str(x).lower() for r in RESOLVED_OUTCOMES) if pd.notna(x) else False)
res = tw.groupby('lsoa21cd').agg(n=('Crime ID', 'count'), r=('resolved', 'sum')).reset_index()
res['resolution_rate'] = (res['r'] / res['n'] * 100).round(2)

d = (LSOAS.merge(agg, on='lsoa21cd', how='left')
          .merge(res[['lsoa21cd', 'resolution_rate']], on='lsoa21cd', how='left')
          .merge(static[['lsoa21cd', 'employment_deprivation', 'ntl_mean_radiance']], on='lsoa21cd', how='left'))
d['crime_count_24'] = d['crime_count_24'].fillna(0)
d['resolution_rate'] = d['resolution_rate'].fillna(d['resolution_rate'].mean())

FEATURES = list(WEIGHTS.keys())
X = d[FEATURES].copy()
for c in LOG_FEATURES:
    X[c] = np.log1p(X[c])
Xn = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=FEATURES, index=d.index)
d['risk_score_24'] = sum(Xn[f] * WEIGHTS[f] for f in FEATURES)
d['risk_score_24'] = ((d['risk_score_24'] - d['risk_score_24'].min()) /
                      (d['risk_score_24'].max() - d['risk_score_24'].min()) * 100).round(2)
print('24-month risk score rebuilt for', len(d), 'LSOAs')

## 4. Ground-truth label + 5. validation vs baseline

In [ ]:
fut = future.groupby('lsoa21cd').size().reset_index(name='crime_count_future')
d = d.merge(fut, on='lsoa21cd', how='left')
d['crime_count_future'] = d['crime_count_future'].fillna(0)
d['high_demand'] = (d['crime_count_future'] >= d['crime_count_future'].quantile(1 - LABEL_PCT)).astype(int)

y = d['high_demand']
auc_index    = roc_auc_score(y, d['risk_score_24'])
auc_baseline = roc_auc_score(y, d['crime_count_24'])
auc_ntl      = roc_auc_score(y, d['ntl_mean_radiance'])
print('--- ROC-AUC: predicting future top-15% crime LSOAs (England) ---')
print(f'  Composite risk index (24mo): {auc_index:.4f}')
print(f'  Baseline: past crime count : {auc_baseline:.4f}')
print(f'  Night-time radiance alone  : {auc_ntl:.4f}')
print(f'  Index vs baseline: {auc_index-auc_baseline:+.4f}  '
      f'({"adds predictive value" if auc_index > auc_baseline else "no gain over naive baseline"})')
report += [f'AUC index={auc_index:.4f}', f'AUC baseline(past crime)={auc_baseline:.4f}', f'AUC radiance={auc_ntl:.4f}']

## 6. Four tiers from three ROC-validated cut-points

In [ ]:
cuts, aucs = {}, {}
for L in TIER_LEVELS:
    yL = (d['crime_count_future'] >= d['crime_count_future'].quantile(1 - L)).astype(int)
    fpr, tpr, thr = roc_curve(yL, d['risk_score_24'])
    k = int(np.argmax(tpr - fpr))
    cuts[L] = float(thr[k]); aucs[L] = roc_auc_score(yL, d['risk_score_24'])
    print(f'  top-{int(L*100):>2}% future crime: AUC={aucs[L]:.3f}  ->  Youden cut score>= {cuts[L]:.1f}  '
          f'(recall={tpr[k]:.2f}, spec={1-fpr[k]:.2f})')
t1, t2, t3 = cuts[0.15], cuts[0.40], cuts[0.70]
if not (t1 >= t2 >= t3):
    print('  NOTE: cuts not monotonic; sorting so tiers nest.')
    t1, t2, t3 = sorted([t1, t2, t3], reverse=True)
print(f'\nTier boundaries: Tier1 >= {t1:.1f} | Tier2 >= {t2:.1f} | Tier3 >= {t3:.1f} | else Tier4')
report += [f'ROC tier cuts: T1>={t1:.1f}, T2>={t2:.1f}, T3>={t3:.1f}',
           f'AUC by level: 15%={aucs[0.15]:.3f}, 40%={aucs[0.40]:.3f}, 70%={aucs[0.70]:.3f}']

In [ ]:
def sup_tier(s):
    if s >= t1: return 1
    if s >= t2: return 2
    if s >= t3: return 3
    return 4
d['supervised_tier'] = d['risk_score_24'].apply(sup_tier)
print('Supervised tier sizes (1=highest demand):')
for t in range(1, 5):
    n = int((d['supervised_tier'] == t).sum())
    sh = d.loc[d['supervised_tier'] == t, 'high_demand'].mean() if n else 0
    print(f'  Tier {t}: {n:>6} LSOAs ({n/len(d)*100:4.1f}%)  | true top-15% hotspot share: {sh*100:4.1f}%')
    report.append(f'Tier{t}: n={n} ({n/len(d)*100:.1f}%), hotspot-share={sh*100:.1f}%')

## 7. ROC curve with the tier cut-points

In [ ]:
matplotlib.rcParams['figure.dpi'] = 120
y15 = d['high_demand']
fpr, tpr, thr = roc_curve(y15, d['risk_score_24'])
fpr_b, tpr_b, _ = roc_curve(y15, d['crime_count_24'])
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#D85A30', lw=2, label=f'Risk index, top-15% test (AUC={auc_index:.2f})')
ax.plot(fpr_b, tpr_b, color='#1D9E75', lw=1.8, ls='-.', label=f'Baseline past crime (AUC={auc_baseline:.2f})')
ax.plot([0,1],[0,1], color='navy', lw=1.3, ls='--', label='Random (0.50)')
for L, col in zip(TIER_LEVELS, ['#D62728', '#FF7F0E', '#2CA02C']):
    yL = (d['crime_count_future'] >= d['crime_count_future'].quantile(1 - L)).astype(int)
    f2, t2_, th2 = roc_curve(yL, d['risk_score_24'])
    k = int(np.argmax(t2_ - f2))
    ax.scatter(f2[k], t2_[k], color=col, s=80, zorder=5, label=f'Tier cut (top {int(L*100)}%) score>= {th2[k]:.0f}')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('Phase 6 (England): ROC-validated tier cut-points')
ax.legend(loc='lower right', fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / '6_roc_tier_cuts.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: 6_roc_tier_cuts.png')

## 8. Compare supervised tiers with K-means (k=4)

In [ ]:
_p5 = P5 / 'phase5_clusters_k4.parquet'
if not _p5.exists():
    print('Phase 5 k=4 output not found; skipping comparison (Phase 6 stands alone as the tiering layer).')
else:
    km = pd.read_parquet(P5 / 'phase5_clusters_k4.parquet')[['lsoa21cd', 'tier']].rename(columns={'tier': 'kmeans_tier'})
    cmp = d[['lsoa21cd', 'supervised_tier']].merge(km, on='lsoa21cd', how='inner')
    ct = pd.crosstab(cmp['supervised_tier'], cmp['kmeans_tier'])
    print('Cross-tab: supervised tier (rows) vs K-means k=4 tier (cols), 1=highest\n')
    print(ct.to_string())
    agree = (cmp['supervised_tier'] == cmp['kmeans_tier']).mean()
    within1 = ((cmp['supervised_tier'] - cmp['kmeans_tier']).abs() <= 1).mean()
    print(f'\nExact tier agreement: {agree*100:.1f}%  | within +/-1 tier: {within1*100:.1f}%')
    report.append(f'Supervised vs K-means k4: {agree*100:.1f}% exact, {within1*100:.1f}% within 1 tier')

## 9. Map the supervised tiers (England)

In [ ]:
g = gpd.read_file(SHP_FILE)
g = g[g['LSOA21CD'].str.startswith('E')][['LSOA21CD', 'geometry']].rename(columns={'LSOA21CD': 'lsoa21cd'})
g = g.merge(d[['lsoa21cd', 'supervised_tier']], on='lsoa21cd', how='left').to_crs(epsg=4326)
fig, ax = plt.subplots(figsize=(13, 15))
labels = {1: 'Tier 1 - High (top 15% cut)', 2: 'Tier 2 - Elevated (top 40%)',
          3: 'Tier 3 - Moderate (top 70%)', 4: 'Tier 4 - Low'}
for t in range(1, 5):
    g[g['supervised_tier'] == t].plot(ax=ax, color=TIER_COLOURS[t], linewidth=0.02, edgecolor='white', alpha=0.85)
um = g[g['supervised_tier'].isna()]
if len(um): um.plot(ax=ax, color='#cccccc', linewidth=0.02, edgecolor='white')
ax.legend(handles=[mpatches.Patch(color=TIER_COLOURS[t], label=labels[t]) for t in range(1, 5)],
          loc='upper left', fontsize=9, framealpha=0.9)
ax.set_title('Supervised (ROC-validated) priority tiers - england_final_light', fontsize=15, fontweight='bold')
ax.set_axis_off(); plt.tight_layout()
plt.savefig(OUT / '6_supervised_tier_map.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: 6_supervised_tier_map.png')

## 10. Save outputs

In [ ]:
keep = ['lsoa21cd', 'crime_count_24', 'resolution_rate', 'employment_deprivation',
        'ntl_mean_radiance', 'risk_score_24', 'crime_count_future', 'high_demand', 'supervised_tier']
d[keep].to_parquet(OUT / 'phase6_supervised_tiers.parquet', index=False)
print('Saved phase6_supervised_tiers.parquet', d[keep].shape)

# --- per-tier summary (Phase 6 is the tiering layer; replaces the old Phase 5 profiles) ---
_featcols = [c for c in keep if c not in ['lsoa21cd','crime_count_future','high_demand','supervised_tier']]
prof = d.groupby('supervised_tier').agg(n_lsoas=('lsoa21cd','size'),
        mean_risk_score=('risk_score_24','mean'), score_min=('risk_score_24','min'),
        score_max=('risk_score_24','max'),
        hotspot_share_pct=('high_demand', lambda x: x.mean()*100)).reset_index()
prof['pct'] = prof['n_lsoas']/len(d)*100
prof = prof.merge(d.groupby('supervised_tier')[_featcols].mean().reset_index(), on='supervised_tier').round(2)
prof.to_csv(OUT / 'phase6_tier_profiles.csv', index=False)
print('Saved phase6_tier_profiles.csv'); print(prof.to_string(index=False))

lines = ['CBL-16 Phase 6 - Supervised Validation & Tiering (england_final_light)', '='*62] + report
(OUT / 'phase6_report.txt').write_text('\n'.join(lines), encoding='utf-8')
print('Saved phase6_report.txt')
print('\n' + '\n'.join(lines))

## 11. Final tiering (Option B): cut on the displayed score

The tiers above were calibrated on the 24-month training score, but the dashboard and report display the 36-month score, which made the tier ranges overlap. This final step keeps the calibrated tier sizes but re-ranks them on the displayed 36-month score, so the tier and the number shown always agree. See OPTION_B_SCORE_CUT_TIERS.md.

In [ ]:
import matplotlib.patches as mpatches
_OUT = BASE / 'england_final_light' / 'outputs' / 'phase6'
_TC = {1:'#D62728',2:'#FF7F0E',3:'#2CA02C',4:'#1F77B4'}
_TL = {1:'Tier 1 - Highest demand',2:'Tier 2 - High demand',3:'Tier 3 - Moderate demand',4:'Tier 4 - Low demand'}
_feats = ['ntl_mean_radiance', 'employment_deprivation', 'resolution_rate']
_p6 = pd.read_parquet(_OUT/'phase6_supervised_tiers.parquet')
_p4 = pd.read_parquet(BASE/'england_final_light'/'outputs'/'phase4'/'phase4_risk_scores.parquet')[['lsoa21cd','risk_score_scaled']+_feats]
_calib = 'supervised_tier' if 'supervised_tier' in _p6.columns else 'tier'
dd = _p6[['lsoa21cd',_calib,'high_demand','risk_score_24']].rename(columns={_calib:'supervised_tier'}).merge(_p4, on='lsoa21cd', how='left')
_sizes = dd['supervised_tier'].value_counts().sort_index()
_order = dd['risk_score_scaled'].rank(ascending=False, method='first')
dd['tier'] = np.searchsorted(_sizes.cumsum().to_numpy(), _order.to_numpy(), side='left') + 1
_moved = int((dd['tier']!=dd['supervised_tier']).sum())
print(f'Option B: {_moved} labels moved ({_moved/len(dd)*100:.1f}%), sizes preserved')
prof = dd.groupby('tier').agg(n_lsoas=('lsoa21cd','size'), mean_risk_score=('risk_score_scaled','mean'),
        score_min=('risk_score_scaled','min'), score_max=('risk_score_scaled','max'),
        hotspot_share_pct=('high_demand', lambda x: x.mean()*100)).reset_index()
prof['pct'] = prof['n_lsoas']/len(dd)*100
prof = prof.merge(dd.groupby('tier')[_feats].mean().reset_index(), on='tier').round(2)
prof.to_csv(_OUT/'phase6_tier_profiles.csv', index=False)
dd[['lsoa21cd','tier','supervised_tier','risk_score_scaled','risk_score_24','high_demand']+_feats].to_parquet(_OUT/'phase6_supervised_tiers.parquet', index=False)
print(prof[['tier','n_lsoas','pct','score_min','score_max','hotspot_share_pct']].to_string(index=False))
g0 = gpd.read_file(BASE/'data'/'Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V5_-7203918579177758597'/'LSOA_2021_EW_BGC_V5.shp')
g0 = g0[g0['LSOA21CD'].str.startswith('E')].rename(columns={'LSOA21CD':'lsoa21cd'})
g = g0.merge(dd[['lsoa21cd','tier']], on='lsoa21cd', how='left').to_crs(epsg=4326)
fig, ax = plt.subplots(figsize=(13,15))
for t in range(1,5): g[g['tier']==t].plot(ax=ax, color=_TC[t], linewidth=0.03, edgecolor='white', alpha=0.85)
_um=g[g['tier'].isna()];
if len(_um): _um.plot(ax=ax, color='#cccccc', linewidth=0.03, edgecolor='white')
ax.legend(handles=[mpatches.Patch(color=_TC[t], label=_TL[t]) for t in range(1,5)], loc='upper left', fontsize=9, framealpha=0.9)
ax.set_title('Priority tiers (displayed-score bands at validated sizes) - england_final_light', fontsize=14, fontweight='bold'); ax.set_axis_off(); plt.tight_layout()
plt.savefig(_OUT/'6_supervised_tier_map.png', dpi=150, bbox_inches='tight'); plt.show(); print('Saved map')
_pf = ['risk_score_scaled']+_feats; _lab = ['Risk\nscore', 'Night\nlights', 'Employ.\ndepriv.', 'Resolution']
_pr = dd.groupby('tier')[_pf].mean(); _nr=(_pr-_pr.min())/(_pr.max()-_pr.min())
_sz=dd['tier'].value_counts().sort_index(); _hot=dd.groupby('tier')['high_demand'].mean()*100
fig, axes = plt.subplots(1,4,figsize=(13.6,4.6),sharey=True)
for t,ax in zip(range(1,5),axes):
    ax.bar(_lab,_nr.loc[t].values,color=_TC[t],edgecolor='white',alpha=0.9)
    n=int(_sz.get(t,0)); ax.set_title(f'Tier {t}\nn={n} ({n/len(dd)*100:.0f}%)\nreal hotspots: {_hot.get(t,0):.0f}%',fontsize=10,fontweight='bold')
    ax.set_ylim(0,1.12); ax.tick_params(axis='x',labelsize=8); ax.spines[['top','right']].set_visible(False)
    if t==1: ax.set_ylabel('Normalised mean (0=low, 1=high)',fontsize=9)
plt.suptitle('Phase 5 tier profiles - england_final_light (displayed-score bands at validated sizes)',fontsize=12,y=1.06)
plt.tight_layout(); plt.savefig(_OUT/'6_tier_profiles.png', dpi=150, bbox_inches='tight'); plt.show(); print('Saved profiles')